In [ ]:
# Instalação das bibliotecas (necessário no Google Colab se ainda não instaladas)
try:
    import xgboost, pandas, sklearn, matplotlib
except Exception:
    !pip install xgboost pandas scikit-learn matplotlib openpyxl --quiet


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier


## 1. Carregamento dos dados

In [ ]:
train = pd.read_csv("conjunto_de_treinamento.csv")
test  = pd.read_csv("conjunto_de_teste.csv")

print("Formas:", train.shape, test.shape)
display(train.head(3))


Formas: (20000, 42) (5000, 41)


,id_solicitante,produto_solicitado,dia_vencimento,forma_envio_solicitacao,tipo_endereco,sexo,idade,estado_civil,qtde_dependentes,grau_instrucao,...,possui_telefone_trabalho,codigo_area_telefone_trabalho,meses_no_trabalho,profissao,ocupacao,profissao_companheiro,grau_instrucao_companheiro,local_onde_reside,local_onde_trabalha,inadimplente
0,1,1,10,presencial,1,M,85,2,0,0,...,N,,0,9.0,1.0,0.0,0.0,600.0,600.0,0
1,2,1,25,internet,1,F,38,1,0,0,...,N,,0,2.0,5.0,NaN,NaN,492.0,492.0,0
2,3,1,20,internet,1,F,37,2,0,0,...,N,,0,NaN,NaN,NaN,NaN,450.0,450.0,1


## 2. Separação da variável alvo e features

In [ ]:
target_col = "inadimplente"

if target_col not in train.columns:
    raise ValueError("Coluna alvo 'inadimplente' não encontrada no conjunto de treinamento.")

y_train = train[target_col].copy()
X_train = train.drop(columns=[target_col]).copy()
X_test  = test.copy()

id_col = "id_solicitante" if "id_solicitante" in test.columns else None
test_ids = test[id_col].copy() if id_col else pd.Series(np.arange(len(test)), name="row_id")

print("Distribuição da classe alvo:")
display(y_train.value_counts(normalize=True).rename("proporcao").to_frame())


Distribuição da classe alvo:


,proporcao
inadimplente,
0,0.5
1,0.5


## 3. Normalização de colunas binárias

In [ ]:
def map_binary_columns(df):
    mapping_sets = {
        "Y/N": {"Y": 1, "N": 0, "y": 1, "n": 0},
        "M/F/N": {"M": 0, "F": 1, "N": -1},
        "0/1": {"0": 0, "1": 1}
    }
    bin_cols_map = {
        "possui_telefone_residencial": "Y/N",
        "possui_telefone_celular": "Y/N",
        "possui_telefone_trabalho": "Y/N",
        "vinculo_formal_com_empresa": "Y/N",
        "sexo": "M/F/N",
    }
    for col, mname in bin_cols_map.items():
        if col in df.columns:
            df[col] = df[col].map(mapping_sets[mname]).astype(float)
    return df

X_train = map_binary_columns(X_train)
X_test  = map_binary_columns(X_test)


## 4. Identificação de colunas numéricas e categóricas

In [ ]:
# Liste des colonnes à supprimer (nuisibles ou identifiants)
cols_to_drop = [
]


X_train = X_train.drop(columns=[c for c in cols_to_drop if c in X_train.columns], errors="ignore")
X_test  = X_test.drop(columns=[c for c in cols_to_drop if c in X_test.columns],  errors="ignore")

print("Colonnes supprimées :", [c for c in cols_to_drop if c not in X_train.columns])
print("Dimensions X_train :", X_train.shape)
print("Dimensions X_test  :", X_test.shape)


cat_manual = [
    "estado_civil",
    "tipo_residencia",
    "profissao",
    "ocupacao",
    "profissao_companheiro",
    "nacionalidade",
    "local_onde_reside",
    "local_onde_trabalha",
]

numeric_features = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
categorical_features = [c for c in X_train.columns if c not in numeric_features]
categorical_features = categorical_features + cat_manual
numeric_features = [c for c in numeric_features if c not in cat_manual]
print(categorical_features)

print("Nº colunas numéricas:", len(numeric_features))
print("Nº colunas categóricas:", len(categorical_features))


Colonnes supprimées : []
Dimensions X_train : (20000, 41)
Dimensions X_test  : (5000, 41)
['forma_envio_solicitacao', 'estado_onde_nasceu', 'estado_onde_reside', 'codigo_area_telefone_residencial', 'estado_onde_trabalha', 'codigo_area_telefone_trabalho', 'estado_civil', 'tipo_residencia', 'profissao', 'ocupacao', 'profissao_companheiro', 'nacionalidade', 'local_onde_reside', 'local_onde_trabalha']
['id_solicitante', 'produto_solicitado', 'dia_vencimento', 'tipo_endereco', 'sexo', 'idade', 'qtde_dependentes', 'grau_instrucao', 'possui_telefone_residencial', 'meses_na_residencia', 'possui_telefone_celular', 'possui_email', 'renda_mensal_regular', 'renda_extra', 'possui_cartao_visa', 'possui_cartao_mastercard', 'possui_cartao_diners', 'possui_cartao_amex', 'possui_outros_cartoes', 'qtde_contas_bancarias', 'qtde_contas_bancarias_especiais', 'valor_patrimonio_pessoal', 'possui_carro', 'vinculo_formal_com_empresa', 'possui_telefone_trabalho', 'meses_no_trabalho', 'grau_instrucao_companheiro'

## 5. Pipeline de pré-processamento base (imputação + OneHotEncoder)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore",
        min_frequency=20,
        sparse_output=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 6. Modelo XGBoost (referência)

In [ ]:
pos_rate = y_train.mean()
scale_pos_weight = (1 - pos_rate) / pos_rate if pos_rate > 0 else 1.0
print(f"Taxa de positivos (1 = inadimplente): {pos_rate:.3f} — scale_pos_weight ≈ {scale_pos_weight:.2f}")

xgb_model = XGBClassifier(
    n_estimators=500,      # 'model__n_estimators': 550,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.75  ,          #'model__subsample': 0.75
    colsample_bytree=0.68,   #'model__colsample_bytree': 0.7
    min_child_weight=5,
    reg_alpha=3,            #'model__reg_alpha': 3,
    reg_lambda=10,          #'model__reg_lambda': 10,
    objective="binary:logistic",
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=float(scale_pos_weight)
)

xgb_clf = Pipeline(steps=[("prep", preprocessor), ("model", xgb_model)])

xgb_scores = cross_val_score(xgb_clf, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
print("AUC XGBoost (5-fold CV):", xgb_scores.mean(), "+/-", xgb_scores.std())


Taxa de positivos (1 = inadimplente): 0.500 — scale_pos_weight ≈ 1.00
AUC XGBoost (5-fold CV): 0.64069925 +/- 0.014351195909008763


## 7. Modelo SVM (LinearSVC)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
import numpy as np

# Pipeline SVM de base
svm_clf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LinearSVC(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

# Petite grille d'hyperparamètres autour de C
param_grid_svm = {
    "model__C": [0.01, 0.1, 1.0, 3.0, 10.0],
    "model__loss": ["hinge", "squared_hinge"],
}

svm_search = GridSearchCV(
    estimator=svm_clf,
    param_grid=param_grid_svm,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

svm_search.fit(X_train, y_train)

print("Meilleurs paramètres SVM :", svm_search.best_params_)
print("Meilleure AUC (CV) SVM   :", svm_search.best_score_)



Fitting 5 folds for each of 10 candidates, totalling 50 fits


KeyboardInterrupt: 

In [ ]:
best_svm = svm_search.best_estimator_

svm_scores = []
for train_idx, val_idx in cv.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    best_svm.fit(X_tr, y_tr)
    scores_val = best_svm.decision_function(X_val)
    svm_scores.append(roc_auc_score(y_val, scores_val))

svm_scores = np.array(svm_scores)
print("AUC SVM (5-fold CV) avec meilleurs hyperparamètres :", svm_scores.mean(), "+/-", svm_scores.std())

y_test_pred = best_svm.predict(X_test)


submission = pd.DataFrame({
    "id_solicitante": test_ids,
    "inadimplente": y_test_pred
})


submission.to_csv("submission_kaggle.csv", index=False)

submission.head()

## 8. Modelo AdaBoost

In [ ]:
ada_clf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2),
        n_estimators=400,
        learning_rate=0.05,
        random_state=42
    ))
])

ada_scores = cross_val_score(ada_clf, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
print("AUC AdaBoost (5-fold CV):", ada_scores.mean(), "+/-", ada_scores.std())

AUC AdaBoost (5-fold CV): 0.612588175 +/- 0.015313023223019981


## 9. Modelo KNN (com escalonamento)

In [ ]:
numeric_transformer_knn = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer_knn = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor_knn = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_knn, numeric_features),
        ("cat", categorical_transformer_knn, categorical_features),
    ]
)

knn_clf = Pipeline(steps=[
    ("prep", preprocessor_knn),
    ("model", KNeighborsClassifier(
        n_neighbors=15,
        weights="distance"
    ))
])

knn_scores = cross_val_score(knn_clf, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
print("AUC KNN (5-fold CV):", knn_scores.mean(), "+/-", knn_scores.std())


AUC KNN (5-fold CV): 0.5732301249999999 +/- 0.007820212971844183


Modelo LGMC

In [ ]:

try:
    from lightgbm import LGBMClassifier
except ImportError:
    !pip install lightgbm --quiet
    from lightgbm import LGBMClassifier

from sklearn.model_selection import cross_val_score


lgbm_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    reg_alpha=1.0,
    objective="binary",
    random_state=42,
    n_jobs=-1
)

lgbm_clf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", lgbm_model)
])


lgbm_scores = cross_val_score(
    lgbm_clf,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("AUC LightGBM (5-fold CV):", lgbm_scores.mean(), "+/-", lgbm_scores.std())



AUC LightGBM (5-fold CV): 0.6297071 +/- 0.012434090910275674


## 10. Treinamento final do melhor modelo e submissão para o Kaggle

In [ ]:
print("\nResumo AUC ROC (5-fold CV):")
print(f"XGBoost : {xgb_scores.mean():.4f}")
#print(f"SVM     : {svm_scores.mean():.4f}")
print(f"AdaBoost: {ada_scores.mean():.4f}")
print(f"KNN     : {knn_scores.mean():.4f}")
print(f"AUC LightGBM (5-fold CV): {lgbm_scores.mean():.4f}")

best_model = xgb_clf  # escolha manual; substitua se outro for melhor

best_model.fit(X_train, y_train)

y_test_proba = best_model.predict_proba(X_test)[:, 1]
y_test_pred  = (y_test_proba >= 0.5).astype(int)

sub = pd.DataFrame({
    id_col if id_col else "row_id": test_ids,
    "inadimplente": y_test_pred
})

sub.to_csv("submission_melhor_modelo_multimodelos.csv", index=False)
sub.head()



Resumo AUC ROC (5-fold CV):
XGBoost : 0.6407
AdaBoost: 0.6126
KNN     : 0.5732
AUC LightGBM (5-fold CV): 0.6297


,id_solicitante,inadimplente
0,20001,1
1,20002,0
2,20003,1
3,20004,0
4,20005,0


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np


param_dist = {

    "model__n_estimators":     [500, 600, 700],
    "model__subsample":        [0.65, 0.7, 0.75],
    "model__colsample_bytree": [0.65, 0.7, 0.75],
    "model__reg_alpha":        [3, 5, 7],
    "model__reg_lambda":       [8, 10, 12],
}

search_light = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    n_iter=12,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

search_light.fit(X_train, y_train)

print("Meilleurs hyperparamètres (affinage léger) :")
print(search_light.best_params_)
print("\nMeilleure AUC (CV) :", search_light.best_score_)


In [ ]:
best_clf = search_light.best_estimator_


from sklearn.model_selection import cross_val_score
scores = cross_val_score(best_clf, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
print("AUC moyen (best_clf) :", scores.mean(), "+/-", scores.std())


In [ ]:
best_clf.fit(X_train, y_train)
y_test_proba = best_clf.predict_proba(X_test)[:, 1]
y_test_pred  = (y_test_proba >= 0.5).astype(int)

sub = pd.DataFrame({
    id_col if id_col else "row_id": test_ids,
    "inadimplente": y_test_pred
})
sub.to_csv("submission_xgb_affinage_leger.csv", index=False)
sub.head()
